<a href="https://colab.research.google.com/github/sabihadudhia/Thesis-Hallucination-Benchmarks/blob/main/Gemma_3_4B_TruthfulQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess, os, re, json, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from huggingface_hub import notebook_login
import warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
torch.manual_seed(42)

notebook_login()

In [2]:
repos = {
    "TruthfulQA": "https://github.com/sylinrl/TruthfulQA.git",
    "HaluEval": "https://github.com/RUCAIBox/HaluEval.git",
    "OpenFActScore": "https://github.com/lflage/OpenFActScore.git",
}

for name, url in repos.items():
    subprocess.run(["git", "clone", url, name])
    commit = subprocess.run(
        ["git", "-C", name, "rev-parse", "HEAD"],
        capture_output=True, text=True
    ).stdout.strip()
    print(f"{name}: {commit}")

TruthfulQA: d71c110897f5d31c5d7f309e7bc316c152f6f031
HaluEval: b7253db3cdaa0ab2c382f92b26b390109174f77e
OpenFActScore: 35c08f1f6137726986da71f151f001366ca683db


In [3]:
ds = load_dataset("truthfulqa/truthful_qa", "multiple_choice")
print(ds)

README.md:   0%|          | 0.00/9.59k [00:00<?, ?B/s]

multiple_choice/validation-00000-of-0000(…): reconstructing file:   0%|          |  0.00B /  271kB            

multiple_choice/validation-00000-of-0000(…): downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['question', 'mc1_targets', 'mc2_targets'],
        num_rows: 817
    })
})


In [4]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.9 MB/s eta 0:00:00


In [5]:
model_name = "google/gemma-3-12b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [6]:
def normalize_response(text):
    return text.strip()

def score_question(sample, model, tokenizer):
    question = sample["question"]
    choices = sample["mc1_targets"]["choices"]
    labels = sample["mc1_targets"]["labels"]

    prompt = f"Question: {question}\nChoices:\n" + "\n".join(
        f"{i+1}. {c}" for i, c in enumerate(choices)
    ) + "\nAnswer with the number of the correct choice only."

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        # temperature intentionally omitted: ignored under greedy decoding (do_sample=False)
    )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    response = normalize_response(response)

    match = re.match(r'\s*(\d+)', response)
    if not match:
        return {"correct": None, "raw_response": response, "reason": "unparseable"}
    answer_idx = int(match.group(1)) - 1
    if answer_idx < 0 or answer_idx >= len(labels):
        return {"correct": None, "raw_response": response, "reason": "out_of_range"}
    return {"correct": labels[answer_idx] == 1, "raw_response": response, "reason": None}

In [7]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/thesis_results', exist_ok=True)

save_path = '/content/drive/MyDrive/thesis_results/gemma3_12b_truthfulqa_mc1_results.jsonl'

results = []
start_time = time.time()

with open(save_path, 'w') as f:
    for i in range(len(ds["validation"])):
        sample = ds["validation"][i]
        result = score_question(sample, model, tokenizer)
        result["question_id"] = i
        results.append(result)
        f.write(json.dumps(result) + "\n")
        if i % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Progress: {i}/{len(ds['validation'])} | Elapsed: {elapsed:.1f}s")

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.1f}s ({total_time/60:.1f} min)")

valid_results = [r for r in results if r["correct"] is not None]
accuracy = sum(r["correct"] for r in valid_results) / len(valid_results)
invalid_count = len(results) - len(valid_results)

print(f"Accuracy: {accuracy:.3f}")
print(f"Invalid/unparseable responses: {invalid_count} / {len(results)}")

Mounted at /content/drive
Progress: 0/817 | Elapsed: 4.8s
Progress: 50/817 | Elapsed: 76.3s
Progress: 100/817 | Elapsed: 145.6s
Progress: 150/817 | Elapsed: 214.1s
Progress: 200/817 | Elapsed: 285.0s
Progress: 250/817 | Elapsed: 352.8s
Progress: 300/817 | Elapsed: 421.1s
Progress: 350/817 | Elapsed: 489.3s
Progress: 400/817 | Elapsed: 562.0s
Progress: 450/817 | Elapsed: 635.0s
Progress: 500/817 | Elapsed: 703.5s
Progress: 550/817 | Elapsed: 770.6s
Progress: 600/817 | Elapsed: 838.2s
Progress: 650/817 | Elapsed: 907.5s
Progress: 700/817 | Elapsed: 976.6s
Progress: 750/817 | Elapsed: 1047.0s
Progress: 800/817 | Elapsed: 1118.7s

Done. Total time: 1140.8s (19.0 min)
Accuracy: 0.571
Invalid/unparseable responses: 5 / 817
